# Step 00: Crawl ViFactCheck Articles (Fresh)

Get URLs from ViFactCheck HuggingFace dataset and crawl articles
with improved caption extraction (figcaption -> alt -> title -> default).

- Source: `tranthaihoa/vifactcheck` (HuggingFace)
- Output: `../data/json/recrawled/news_data_vifactcheck_*_recrawled.json` (NEW files)
- Images: `../data/jpg/` (downloads new images)
- Does NOT overwrite existing cleaned files

In [1]:
import subprocess
import sys

for pkg in ["datasets", "httpx", "beautifulsoup4", "lxml", "nest-asyncio", "tqdm", "loguru", "pillow-avif-plugin"]:
    try:
        mod = pkg.replace("-", "_")
        if mod == "pillow_avif_plugin":
            mod = "pillow_avif"
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"Installed {pkg}")

Installed beautifulsoup4


In [2]:
import sys
import os
import json
import nest_asyncio
from pathlib import Path
from collections import Counter

nest_asyncio.apply()
os.environ["OPENSSL_CONF"] = "openssl.cnf"

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [3]:
# Configuration
CONFIG = {
    "dataset_name": "tranthaihoa/vifactcheck",
    "url_column": "Url",
    "splits": ["train", "dev", "test"],
    "output_dir": str(project_root / "notebooks" / "data" / "json" / "recrawled"),
    "cache_dir": str(project_root / "notebooks" / "data" / "caches_recrawl"),
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["cache_dir"], exist_ok=True)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Configuration:
  dataset_name: tranthaihoa/vifactcheck
  url_column: Url
  splits: ['train', 'dev', 'test']
  output_dir: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\json\recrawled
  cache_dir: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\caches_recrawl


In [4]:
# Get URLs from ViFactCheck HuggingFace
from src.processing.dataset_handler import DatasetHandler

dataset_handler = DatasetHandler(CONFIG["dataset_name"])

split_urls = {}
for split in CONFIG["splits"]:
    urls = dataset_handler.get_urls_from_split(
        split=split,
        url_column=CONFIG["url_column"],
    )
    # Deduplicate
    urls = list(set(urls))
    split_urls[split] = urls
    print(f"  {split}: {len(urls)} unique URLs")

print(f"\nTotal: {sum(len(v) for v in split_urls.values())} URLs to crawl")

--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---


--- Extracting URLs from the 'Url' column... ---
--- Found 5062 URLs to crawl. ---
  train: 1035 unique URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 723 URLs to crawl. ---
  dev: 496 unique URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 1447 URLs to crawl. ---
  test: 758 unique URLs

Total: 2289 URLs to crawl


In [5]:
# Force reload module to pick up code changes
import importlib
import src.helpers.httpx_client as hc_mod
import src.crawler.base_crawler as bc_mod
import src.crawler.crawler_factory as cf_mod
importlib.reload(hc_mod)
importlib.reload(bc_mod)
importlib.reload(cf_mod)
from src.crawler.crawler_factory import CrawlerFactory

# Crawl settings
MAX_CONCURRENT = 30
RETRY_FAILED = False  # Set True to retry failed URLs from previous run

async def crawl_split(split, urls):
    print(f"\n{'='*50}")
    print(f"Crawling {split}: {len(urls)} URLs (concurrency={MAX_CONCURRENT}, retry_failed={RETRY_FAILED})")
    print(f"{'='*50}")

    factory = CrawlerFactory(
        cache_filename=os.path.join(CONFIG["cache_dir"], f"cache_{split}.json"),
        failed_log_filename=os.path.join(CONFIG["cache_dir"], f"failed_{split}.json"),
    )

    output_filename = f"news_data_vifactcheck_{split}_recrawled.json"
    await factory.crawl_and_save_all(
        urls, output_filename,
        format_name="custom",
        max_concurrent=MAX_CONCURRENT,
        retry_failed=RETRY_FAILED,
        save_interval=50,
    )
    return output_filename

results = {}
for split, urls in split_urls.items():
    filename = await crawl_split(split, urls)
    results[split] = filename
    print(f"{split}: done -> {filename}")


Crawling train: 1035 URLs (concurrency=30, retry_failed=False)
2026-04-16 23:34:11 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:133 - Skipping 0 previously failed URLs (use retry_failed=True to retry).
2026-04-16 23:34:11 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:138 - URLs to crawl: 1035 (skipped: 0, concurrency=30)


Crawling:  16%|█▌        | 166/1035 [00:15<01:22, 10.50it/s]

2026-04-16 23:34:27 | ERROR    | helpers.httpx_client:_request:101 - An error occurred while requesting URL('https://thanhnien.vn/tranh-cai-chuyen-khong-duoc-xung-ho-may-tao-185230329141726836.htm'): 


Crawling:  27%|██▋       | 283/1035 [00:26<00:53, 14.05it/s]

2026-04-16 23:34:37 | ERROR    | helpers.httpx_client:_request:101 - An error occurred while requesting URL('https://nld.com.vn/giao-duc-khoa-hoc/bo-truong-nguyen-kim-son-dak-lak-can-tiep-tuc-nang-cao-chat-luong-giao-duc-dan-toc-20230323192716517.htm'): 


...

2026-04-16 23:36:11 | ERROR    | helpers.httpx_client:_request:101 - An error occurred while requesting URL('https://baochinhphu.vn/can-xay-dung-chien-luoc-tong-the-dau-tu-phat-trien-cua-cac-tap-doan-tong-cong-ty-102230318113344801.htm'): 


Crawling:  75%|███████▍  | 371/496 [00:29<00:09, 13.56it/s]

2026-04-16 23:36:12 | ERROR    | helpers.httpx_client:_request:101 - An error occurred while requesting URL('https://thanhnien.vn/tong-thong-putin-chi-ten-nuoc-dung-sau-vu-no-duong-ong-nord-stream-185230326075245785.htm'): 


Crawling: 100%|██████████| 496/496 [00:40<00:00, 12.19it/s]

2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:213 - Saved 492 new + 496 existing = 988 total articles to data\json\news_data_vifactcheck_dev_recrawled.json
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:227 - 
--- Crawling Summary ---
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:228 - Completed (total): 492 URLs
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:229 - Failed this run:   4 URLs
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:230 - Failed (total):    4 URLs
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:236 - Failure reasons:
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_factory:_print_summary:238 -      4x Unknown error
dev: done -> news_data_vifactcheck_dev_recrawled.json

Crawling test: 758 URLs (concurrency=30, retry_failed=False)
2026-04-16 23:36:23 | INFO     | src.crawler.crawler_fact

In [6]:
# Move output files to recrawled directory
# CrawlerFactory saves relative to CWD, find and move them
import shutil
import glob

dst_dir = Path(CONFIG["output_dir"])
dst_dir.mkdir(parents=True, exist_ok=True)

# Search all possible locations where CrawlerFactory might have saved
search_dirs = [
    Path(".") / "data" / "json",                          # CWD/data/json/
    project_root / "data" / "json",                       # project_root/data/json/
    project_root / "notebooks" / "data" / "json",         # notebooks/data/json/
    project_root / "notebooks" / "workflow_coolant" / "data" / "json",  # workflow_coolant/data/json/
]

for split in CONFIG["splits"]:
    filename = f"news_data_vifactcheck_{split}_recrawled.json"
    dst_path = dst_dir / filename

    if dst_path.exists():
        print(f"{filename}: already in destination")
        continue

    found = False
    for search_dir in search_dirs:
        src_path = search_dir / filename
        if src_path.exists() and src_path.resolve() != dst_path.resolve():
            shutil.move(str(src_path), str(dst_path))
            print(f"{filename}: moved from {search_dir}")
            found = True
            break

    if not found:
        print(f"{filename}: NOT FOUND in any location")

print(f"\nFiles in {dst_dir}:")
for f in sorted(dst_dir.glob("*.json")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

news_data_vifactcheck_train_recrawled.json: moved from data\json
news_data_vifactcheck_dev_recrawled.json: moved from data\json
news_data_vifactcheck_test_recrawled.json: moved from data\json

Files in g:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\json\recrawled:
  news_data_vifactcheck_dev_recrawled.json: 4.9 MB
  news_data_vifactcheck_test_recrawled.json: 4.0 MB
  news_data_vifactcheck_train_recrawled.json: 16.0 MB


In [7]:
# Verify results: caption coverage
DEFAULT_CAPTION = "Không được đề cập"
dst_dir = Path(CONFIG["output_dir"])

print("Caption Coverage in Recrawled Data:")
print(f"{'Split':<8} {'Articles':>10} {'Images':>8} {'Real Cap':>10} {'Default':>10} {'%Real':>8}")
print("-" * 60)

for split in CONFIG["splits"]:
    path = dst_dir / f"news_data_vifactcheck_{split}_recrawled.json"
    if not path.exists():
        print(f"{split}: file not found")
        continue

    with open(path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    n_articles = len(articles)
    total_imgs = sum(len(a.get("images", [])) for a in articles)
    real_caps = sum(
        1 for a in articles for img in a.get("images", [])
        if (img.get("caption") or "").strip()
        and (img.get("caption") or "").strip() != DEFAULT_CAPTION
    )
    default_caps = total_imgs - real_caps
    pct = real_caps / max(total_imgs, 1) * 100
    print(f"{split:<8} {n_articles:>10} {total_imgs:>8} {real_caps:>10} {default_caps:>10} {pct:>7.1f}%")

print(f"\nNext: Run 0_extract_pairs.ipynb to extract (image, caption) pairs")

Caption Coverage in Recrawled Data:
Split      Articles   Images   Real Cap    Default    %Real
------------------------------------------------------------
train          3090     4164       4098         66    98.4%
dev             988     1226       1216         10    99.2%
test            758     1033       1018         15    98.5%

Next: Run 0_extract_pairs.ipynb to extract (image, caption) pairs
